# freqk_gr — ΔAF analysis (field vs seed-mix founder)

Staged pipeline with checkpoints — each stage saves to disk so you only recompute once:

| Stage | Output file |
|-------|-------------|
| 1. Seed-mix baseline | `results/seedmix_baseline.tsv` |
| 2. Variant positions | `results/variant_positions.tsv` |
| 3. Field samples Δp  | `results/field_samples_delta_p.tsv` |
| 4. Wide Δp matrix   | `results/dp_wide.parquet` |


In [7]:
import gzip
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ── Auto-detect environment ───────────────────────────────────────────────────
_LOCAL  = Path("/Users/tatiana/Documents_new/freqk_gr")
_CLUSTER = Path("/home/tbellagio/scratch/freqk_gr")

if _LOCAL.exists():
    PROJECT = _LOCAL
    PANG    = Path("/Users/tatiana/Documents_new/pang")
    print("Running locally (sshfs mount)")
else:
    PROJECT = _CLUSTER
    PANG    = Path("/home/tbellagio/scratch/pang")
    print("Running on cluster")

RESULTS = PROJECT / "results"
DATA    = PROJECT / "data"
VCF     = PANG / "sv_panel/merge_vcfs/panel.snp_ins_del.vcf.gz"
K = 31
N_SAMPLE = 50_000   # variants to draw for EDA plots; set None for all

# ── checkpoint paths ──────────────────────────────────────────────────────────
SM_BASELINE_FILE  = RESULTS / "seedmix_baseline.tsv"
VAR_POS_FILE      = RESULTS / "variant_positions.tsv"
FIELD_DELTA_FILE  = RESULTS / "field_samples_delta_p.tsv"
DP_WIDE_FILE      = RESULTS / "dp_wide.parquet"

print(f"PROJECT : {PROJECT}")
print(f"RESULTS : {RESULTS}")
print(f"VCF     : {VCF}")


'/Users/tatiana/Documents_new/freqk_gr/summaries'

In [1]:
import gzip
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

RESULTS = Path("/home/tbellagio/scratch/freqk_gr/results")
DATA    = Path("/home/tbellagio/scratch/freqk_gr/data")
K = 31
N_SAMPLE = 50_000   # variants to draw for EDA plots; set None for all

# ── checkpoint paths ──────────────────────────────────────────────────────────
SM_BASELINE_FILE  = RESULTS / "seedmix_baseline.tsv"
VAR_POS_FILE      = RESULTS / "variant_positions.tsv"
FIELD_DELTA_FILE  = RESULTS / "field_samples_delta_p.tsv"
DP_WIDE_FILE      = RESULTS / "dp_wide.parquet"


---
## Stage 1 — Seed-mix baseline

Average AF across S1–S8 per variant (infinite-N assumption for founder pool).  
Requires ≥ `MIN_SM_REPS` replicates to have k-mer coverage at a variant.

**Saves →** `results/seedmix_baseline.tsv`

In [2]:
MIN_SM_REPS = 4

def load_counts(path):
    df = pd.read_csv(path, sep="|", header=None, names=["ref","alt"], dtype=float)
    df[["ref","alt"]] = df[["ref","alt"]].fillna(0).astype(int)
    df["variant_idx"] = df.index
    total = df["ref"] + df["alt"]
    df["af"] = np.where(total > 0, df["alt"] / total, np.nan)
    df["coverage_kmers"] = total
    return df

sm_files = sorted(RESULTS.glob(f"seedmix/*/k{K}/*.counts_by_allele.k{K}.tsv"))
print(f"Seed-mix replicates found: {len(sm_files)}")
for f in sm_files:
    print(" ", f.parent.parent.name)


Seed-mix replicates found: 0


In [3]:
sm_dfs = []
for f in sm_files:
    tmp = load_counts(f)
    tmp["replicate"] = f.parent.parent.name
    sm_dfs.append(tmp[tmp["coverage_kmers"] > 0][["variant_idx","replicate","af"]])

sm_all = pd.concat(sm_dfs, ignore_index=True)

sm_baseline = (
    sm_all.groupby("variant_idx")["af"]
    .agg(af_sm="mean", af_sm_std="std", n_sm_reps="count")
    .reset_index()
)
sm_baseline = sm_baseline[sm_baseline["n_sm_reps"] >= MIN_SM_REPS].copy()

print(f"Baseline variants (≥{MIN_SM_REPS}/8 reps covered): {len(sm_baseline):,}")
print(f"Mean cross-replicate std: {sm_baseline['af_sm_std'].mean():.4f}")

sm_baseline.to_csv(SM_BASELINE_FILE, sep="\t", index=False)
print(f"Saved → {SM_BASELINE_FILE}")
sm_baseline.head()


ValueError: No objects to concatenate

---
## Stage 2 — Variant positions

Load `variant_positions.k31.tsv` saved by the pipeline (chrom, pos — correct  
post-dedup mapping). Annotate with var_type (SNP/INS/DEL) and sv_size from the VCF.

**Saves →** `results/variant_positions.tsv`

In [ ]:
pos_file = next(RESULTS.glob(f"*/*/k{K}/*.variant_positions.k{K}.tsv"), None)
if pos_file is None:
    raise FileNotFoundError("No variant_positions file found — rerun pipeline with updated run_freqk.sh")
print(f"Positions file: {pos_file}")

vcf_pos = pd.read_csv(pos_file, header=None, names=["chrom","pos"], sep=",",
                      dtype={"chrom":str,"pos":int})
vcf_pos.index.name = "variant_idx"
vcf_pos = vcf_pos.reset_index()
print(f"Variants in deduplicated index: {len(vcf_pos):,}")


In [ ]:
# Annotate with var_type and sv_size from the VCF (lookup by chrom+pos)
VCF = VCF

vcf_lookup = {}
with gzip.open(VCF, "rt") as fh:
    for line in fh:
        if line.startswith("#"):
            continue
        parts = line.split("\t", 5)
        key = (parts[0], int(parts[1]))
        if key not in vcf_lookup:
            vcf_lookup[key] = (len(parts[3]), len(parts[4].strip()))

ref_len = vcf_pos.apply(lambda r: vcf_lookup.get((r.chrom, r.pos), (1,1))[0], axis=1)
alt_len = vcf_pos.apply(lambda r: vcf_lookup.get((r.chrom, r.pos), (1,1))[1], axis=1)

vcf_pos["var_type"] = "SNP"
vcf_pos.loc[alt_len > ref_len, "var_type"] = "INS"
vcf_pos.loc[ref_len > alt_len, "var_type"] = "DEL"
vcf_pos["sv_size"] = (alt_len - ref_len).abs()

print(vcf_pos["var_type"].value_counts())
vcf_pos.to_csv(VAR_POS_FILE, sep="\t", index=False)
print(f"Saved → {VAR_POS_FILE}")
vcf_pos.head()


---
## Stage 3 — Field samples Δp

Load all field sample counts, filter to variants with seed-mix baseline,  
compute Δp = field AF − seed-mix AF, join sample metadata.

**Saves →** `results/field_samples_delta_p.tsv`

In [ ]:
def parse_sample_id(sid):
    return sid[4:6], sid[6:8], sid[8:16], sid[8:12]  # site, plot, date, year

sm_idx = set(sm_baseline["variant_idx"])

af_files = sorted(RESULTS.glob(f"*/*/k{K}/*.counts_by_allele.k{K}.tsv"))
af_files = [f for f in af_files if "seedmix" not in f.parts]
print(f"Field samples: {len(af_files)}")


In [ ]:
rows = []
for path in af_files:
    sid = path.parent.parent.name
    site, plot, date, year = parse_sample_id(sid)
    df = load_counts(path)
    df = df[(df["coverage_kmers"] > 0) & (df["variant_idx"].isin(sm_idx))].copy()
    df["sample_id"] = sid
    df["site"]      = site
    df["plot"]      = plot
    df["date"]      = pd.to_datetime(date, format="%Y%m%d")
    df["year"]      = year
    rows.append(df[["variant_idx","sample_id","site","plot","date","year","af","coverage_kmers"]])
    print(f"  {sid}: {len(df):,} variants")

df_tall = pd.concat(rows, ignore_index=True)
print(f"\nTotal: {df_tall.shape}")


In [ ]:
# Join sample metadata
meta = pd.read_csv(DATA / "samples_data_fix57.csv", usecols=[
    "sampleid","flowerscollected","generation","fix_57_generation",
    "coverage","weighted_mean_coverage","usesample"
]).rename(columns={"sampleid":"sample_id"})

df_tall = df_tall.merge(meta, on="sample_id", how="left")
df_tall["N"]      = df_tall["flowerscollected"]
df_tall["min_af"] = 1 / df_tall["N"]

# Join seed-mix baseline and compute Δp
df_tall = df_tall.merge(
    sm_baseline[["variant_idx","af_sm","af_sm_std","n_sm_reps"]],
    on="variant_idx", how="left"
)
df_tall["delta_p"]     = df_tall["af"] - df_tall["af_sm"]
df_tall["abs_delta_p"] = df_tall["delta_p"].abs()

df_tall.to_csv(FIELD_DELTA_FILE, sep="\t", index=False)
print(f"Saved → {FIELD_DELTA_FILE}  ({df_tall.shape})")
df_tall.head()


---
## Stage 4 — Wide Δp matrix

Pivot to variants × samples, annotate rows with chrom/pos/var_type/sv_size,  
drop all-zero/all-NaN rows.

**Saves →** `results/dp_wide.parquet`

In [ ]:
dp_wide = df_tall.pivot_table(
    index="variant_idx", columns="sample_id", values="delta_p"
)
dp_wide.columns.name = None

# Drop rows that are entirely NaN or 0 (no signal)
mask = (dp_wide.fillna(0) != 0).any(axis=1)
dp_wide = dp_wide[mask]
print(f"Wide matrix after filtering: {dp_wide.shape}")

# Annotate with chrom, pos, var_type, sv_size
dp_annotated = (
    dp_wide
    .reset_index()
    .merge(vcf_pos[["variant_idx","chrom","pos","var_type","sv_size"]], on="variant_idx", how="left")
    .set_index(["chrom","pos"])
    .drop(columns="variant_idx")
)

print(f"Chromosomes: {sorted(dp_annotated.reset_index()['chrom'].unique())}")
print(f"Memory: {dp_annotated.memory_usage(deep=True).sum()/1e6:.1f} MB")

dp_annotated.to_parquet(DP_WIDE_FILE)
print(f"Saved → {DP_WIDE_FILE}")
dp_annotated.head()


---
## Analysis — load from checkpoints

Start here if stages 1–4 have already been run.

In [ ]:
dp_annotated = pd.read_parquet(DP_WIDE_FILE)
df_tall      = pd.read_csv(FIELD_DELTA_FILE, sep="\t", parse_dates=["date"])

print(f"dp_annotated: {dp_annotated.shape}")
print(f"df_tall:      {df_tall.shape}")
print(f"Chromosomes:  {sorted(dp_annotated.reset_index()['chrom'].unique())}")


In [ ]:
# Sample N_SAMPLE random variants for EDA plots
rng = np.random.default_rng(42)
sample_chrom_pos = (
    dp_annotated.reset_index()[["chrom","pos"]]
    .drop_duplicates()
)
if N_SAMPLE and N_SAMPLE < len(sample_chrom_pos):
    idx = rng.choice(len(sample_chrom_pos), size=N_SAMPLE, replace=False)
    sampled_keys = sample_chrom_pos.iloc[idx].set_index(["chrom","pos"]).index
    dp_sample  = dp_annotated.loc[dp_annotated.index.isin(sampled_keys)]
    tall_sample = df_tall[df_tall["variant_idx"].isin(
        dp_annotated.reset_index().merge(
            sample_chrom_pos.iloc[idx], on=["chrom","pos"]
        )["variant_idx"] if "variant_idx" in dp_annotated.reset_index().columns else []
    )]
    print(f"Sampled {len(dp_sample):,} / {len(dp_annotated):,} variants for EDA")
else:
    dp_sample   = dp_annotated
    tall_sample = df_tall
    print("Using full matrix")


## QC

In [ ]:
qc = (
    df_tall.groupby("sample_id")
    .apply(lambda x: pd.Series({
        "n_variants":  len(x),
        "mean_cov":    x["coverage_kmers"].mean(),
        "mean_abs_dp": x["abs_delta_p"].mean(),
    }))
    .reset_index()
)
print(qc.to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for sid, grp in df_tall.groupby("sample_id"):
    grp["delta_p"].dropna().sample(min(10000, len(grp)), random_state=42)        .plot.hist(bins=60, alpha=0.3, density=True, label=sid, ax=ax)
ax.axvline(0, color="black", lw=1, ls="--")
ax.set_xlabel("Δp  (field − seed mix)")
ax.set_ylabel("Density")
ax.set_title("Δp distribution per sample (10k variant sample)")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()


## Δp over time — site 04

In [4]:
site04 = df_tall[df_tall["site"] == "04"].copy()
site04["generation"] = site04["generation"].astype("Int64")

summary = (
    site04.groupby(["date","generation"])["delta_p"]
    .agg(mean="mean", std="std")
    .reset_index()
)

gen_colors = {1:"steelblue", 2:"darkorange", 3:"forestgreen"}

fig, ax = plt.subplots(figsize=(8, 4))
for gen, grp in summary.groupby("generation"):
    ax.errorbar(grp["date"], grp["mean"], yerr=grp["std"],
                fmt="o-", capsize=4, color=gen_colors.get(gen,"gray"),
                label=f"Gen {int(gen)}")
ax.axhline(0, color="black", lw=1, ls="--", label="seed mix")
ax.set_xlabel("Date")
ax.set_ylabel("Mean Δp")
ax.set_title("Site 04 — mean Δp over time")
ax.legend(title="Generation")
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%Y-%m-%d"))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


NameError: name 'df_tall' is not defined

## Per-plot Δp — site 04

In [5]:
site04["date_gen"] = site04.apply(
    lambda r: (
        f"{r['date'].strftime('%Y-%m-%d')}\n(Gen {int(r['generation'])})"
        if pd.notna(r["generation"]) else r["date"].strftime("%Y-%m-%d")
    ), axis=1
)

fig, ax = plt.subplots(figsize=(11, 4))
sns.boxplot(data=site04, x="date_gen", y="delta_p", hue="plot",
            ax=ax, flierprops=dict(marker=".", alpha=0.2, markersize=1))
ax.axhline(0, color="black", lw=1, ls="--")
ax.set_xlabel("Date (Generation)")
ax.set_ylabel("Δp  (field − seed mix)")
ax.set_title("Site 04 — Δp per plot per time point")
ax.legend(title="Plot", bbox_to_anchor=(1.01,1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()


NameError: name 'site04' is not defined

## Filter to variants that changed

In [6]:
DELTA_MIN = 0.05

df_tall["threshold"]  = np.maximum(df_tall["min_af"].fillna(DELTA_MIN), DELTA_MIN)
df_tall["is_changed"] = df_tall["abs_delta_p"] > df_tall["threshold"]

changed_idx = df_tall.groupby("variant_idx")["is_changed"].any()
changed_idx = changed_idx[changed_idx].index

print(f"Total tested variants : {df_tall['variant_idx'].nunique():,}")
print(f"Changed in ≥1 sample  : {len(changed_idx):,}  ({100*len(changed_idx)/df_tall['variant_idx'].nunique():.1f}%)")

dp_changed = dp_annotated[dp_annotated.reset_index()["chrom"].isin(
    dp_annotated.reset_index()["chrom"].unique()   # placeholder — filter by var_type or chrom
)]
print(dp_annotated[["var_type","sv_size"]].value_counts("var_type"))


NameError: name 'df_tall' is not defined